# 2단계 — EfficientNet-B4 학습·공식 Test·ONNX (Kaggle 무료 GPU)

1단계 비공개 Notebook Output의 얼굴 crop으로 모델을 학습한다. Validation에서 프레임 수·통합 방식·기준값을 먼저 고정하고, 그 뒤 공식 Test와 촬영 열화 조건을 평가한다.

## 실행 전

1. 이 Notebook을 `Private`로 유지한다.
2. `Add Input`에서 1단계 Notebook Output을 연결한다.
3. GPU와 Internet을 켠다. Internet은 ImageNet 사전학습 가중치 다운로드에 필요하다.
4. 확인값을 `True`로 변경한다.

In [ ]:
# 1. 설정과 비공개 입력 확인
REPO_URL = "https://github.com/Chunbae-A/face-image.git"
BRANCH = "exp/15-celebdf-deepfake-baseline"
CODE_SOURCE = "embedded"
PREPROCESS_ARCHIVE_PATH = ""  # 비우면 /kaggle/input에서 자동 탐색
I_CONFIRM_PRIVATE_PREPROCESS_OUTPUT_MAY_BE_USED = False
RUN_TRAINING = True
RUN_FINAL_OFFICIAL_TEST = True
SEED = 20260807
EPOCHS = 8
BATCH_SIZE = 8

import os
from pathlib import Path
import sys
IN_KAGGLE = Path("/kaggle").exists() and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if not I_CONFIRM_PRIVATE_PREPROCESS_OUTPUT_MAY_BE_USED:
    raise PermissionError("1단계 얼굴 crop Output을 비공개 연구에 사용할 수 있는지 확인하세요.")
print({"kaggle": IN_KAGGLE, "seed": SEED, "epochs": EPOCHS})

In [ ]:
# 2. ONNX 내보내기·CPU 시험 의존성
# Kaggle 기본 torchvision은 Pillow 12.x와 충돌할 수 있으므로 호환 버전을 고정한다.
%pip install -q --no-cache-dir "onnx==1.18.0" "onnxruntime==1.23.2" "Pillow==11.3.0"

In [ ]:
# 3. 실행 코드 준비 — GitHub 권한이 필요 없는 내장 코드가 기본값
import base64
import os
from pathlib import Path
import subprocess

EMBEDDED_FILES_B64 = {'scripts/celebdf_deepfake.py': '...', 'scripts/run_celebdf_deepfake.py': '...'}
EMBEDDED_CODE_SHA256 = "1f98e8f74539b63eeb5c842d41d47ea94badf0985fc34c451e27261d26337a0b"

if IN_KAGGLE and CODE_SOURCE == "github":
    REPO_DIR = Path("/kaggle/temp/face-image")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    CODE_VERSION = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/temp/face-image")
    for relative_path, encoded in EMBEDDED_FILES_B64.items():
        target = REPO_DIR / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(encoded))
    CODE_VERSION = f"embedded:{EMBEDDED_CODE_SHA256[:12]}"
else:
    REPO_DIR = Path.cwd()
    CODE_VERSION = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()

os.chdir(REPO_DIR)
print({"repo": str(REPO_DIR), "code_source": CODE_SOURCE, "code_version": CODE_VERSION})

In [ ]:
# 4. 1단계 비공개 Output 복원
import json
import shutil

if IN_KAGGLE:
    candidates = (
        [Path(PREPROCESS_ARCHIVE_PATH).expanduser()]
        if PREPROCESS_ARCHIVE_PATH.strip()
        else sorted(Path("/kaggle/input").rglob("celebdf_deepfake_preprocess_private.tar"))
    )
else:
    candidates = [Path(PREPROCESS_ARCHIVE_PATH).expanduser()]
exact = [path for path in candidates if path.is_file()]
if len(exact) != 1:
    raise FileNotFoundError(f"1단계 private TAR 하나가 필요합니다: {exact}")

WORK_ROOT = Path("/kaggle/temp/celebdf_deepfake_train") if IN_KAGGLE else REPO_DIR / "outputs" / "celebdf_deepfake_train"
WORK_ROOT.mkdir(parents=True, exist_ok=True)
subprocess.run(["tar", "-xf", str(exact[0]), "-C", str(WORK_ROOT)], check=True)
CROP_ROOT = WORK_ROOT / "crops"
CROP_MANIFEST = WORK_ROOT / "crop_private_manifest.csv"
INVENTORY = WORK_ROOT / "inventory_aggregate.json"
PREPROCESS_REPORT = WORK_ROOT / "preprocess_aggregate.json"
if not CROP_ROOT.is_dir() or not CROP_MANIFEST.is_file():
    raise RuntimeError("1단계 얼굴 crop 또는 manifest 복원에 실패했습니다.")

OUTPUT_ROOT = Path("/kaggle/working") if IN_KAGGLE else WORK_ROOT / "output"
PRIVATE_MODEL_ROOT = WORK_ROOT / "private_model"
SANITIZED_ROOT = OUTPUT_ROOT / "sanitized"
PRIVATE_MODEL_ROOT.mkdir(parents=True, exist_ok=True)
SANITIZED_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT = PRIVATE_MODEL_ROOT / "efficientnet_b4_best.pt"
ONNX_MODEL = PRIVATE_MODEL_ROOT / "efficientnet_b4.onnx"
TRAIN_REPORT = SANITIZED_ROOT / "train_aggregate.json"
METRICS = SANITIZED_ROOT / "aggregate_metrics.json"
ONNX_EXPORT_REPORT = SANITIZED_ROOT / "onnx_export.json"
ONNX_SMOKE_REPORT = SANITIZED_ROOT / "onnx_cpu_smoke.json"
PRIVATE_SCORES = WORK_ROOT / "frame_scores_private.csv"
FIGURE = SANITIZED_ROOT / "deepfake_baseline_summary.png"
MODEL_CARD = SANITIZED_ROOT / "MODEL_CARD.md"

print({
    "private_stage1_input": str(exact[0]),
    "crop_files": sum(1 for _ in CROP_ROOT.rglob("*.jpg")),
    "runtime_free_gb": round(shutil.disk_usage(WORK_ROOT).free / 1e9, 2),
})

In [ ]:
# 5. PyTorch GPU 확인
import torch
import torchvision

print({
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "cuda_available": torch.cuda.is_available(),
    "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
if IN_KAGGLE and not torch.cuda.is_available():
    raise RuntimeError("Kaggle Settings에서 GPU를 선택하고 다시 시작하세요.")
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
))

In [ ]:
# 6. EfficientNet-B4 학습 — 공식 Test는 사용하지 않음
if not RUN_TRAINING:
    raise ValueError("완료 checkpoint 입력을 별도로 연결하지 않았으므로 RUN_TRAINING=True가 필요합니다.")
subprocess.run([
    sys.executable, "scripts/run_celebdf_deepfake.py", "train",
    "--crop-manifest", str(CROP_MANIFEST), "--crop-root", str(CROP_ROOT),
    "--checkpoint", str(CHECKPOINT), "--train-report", str(TRAIN_REPORT),
    "--input-size", "380", "--train-frames-per-video", "16",
    "--batch-size", str(BATCH_SIZE), "--gradient-accumulation-steps", "2",
    "--epochs", str(EPOCHS), "--early-stopping-patience", "3",
    "--seed", str(SEED), "--require-cuda",
], check=True)
train_report = json.loads(TRAIN_REPORT.read_text(encoding="utf-8"))
assert train_report["official_test_used_for_training"] is False
print({
    "epochs_completed": train_report["epochs_completed"],
    "best_validation_video_auc": train_report["best_validation_video_auc"],
    "checkpoint_sha256": train_report["checkpoint_sha256"],
})

In [ ]:
# 7. Validation 선택을 고정한 뒤 공식 Test·열화 평가
if not RUN_FINAL_OFFICIAL_TEST:
    raise ValueError("공식 Test 실행을 승인할 때 RUN_FINAL_OFFICIAL_TEST=True로 바꾸세요.")
subprocess.run([
    sys.executable, "scripts/run_celebdf_deepfake.py", "evaluate",
    "--crop-manifest", str(CROP_MANIFEST), "--crop-root", str(CROP_ROOT),
    "--checkpoint", str(CHECKPOINT), "--private-scores", str(PRIVATE_SCORES),
    "--metrics", str(METRICS), "--input-size", "380", "--batch-size", "16",
    "--seed", str(SEED), "--target-fpr", "0.01",
    "--frame-counts", "8", "16", "32",
    "--conditions", "clean", "jpeg_q30", "gaussian_blur_sigma2", "low_light_gamma2", "downscale_0_25",
], check=True)
metrics = json.loads(METRICS.read_text(encoding="utf-8"))
print({
    "selected_frames": metrics["selected_frames_per_video"],
    "selected_aggregation": metrics["selected_aggregation"],
    "threshold": metrics["selected_threshold"],
    "official_test_auc": metrics["test_video"]["roc_auc"],
    "official_test_fpr": metrics["test_video"]["fpr"],
    "official_test_recall": metrics["test_video"]["recall"],
    "coverage": metrics["coverage"]["official_test_coverage"],
    "research_gate_pass": metrics["research_gate"]["overall_pass"],
})

In [ ]:
# 8. ONNX 내보내기와 CPU 추론 연결 시험
subprocess.run([
    sys.executable, "scripts/run_celebdf_deepfake.py", "export-onnx",
    "--checkpoint", str(CHECKPOINT), "--output", str(ONNX_MODEL),
    "--report", str(ONNX_EXPORT_REPORT),
], check=True)
subprocess.run([
    sys.executable, "scripts/run_celebdf_deepfake.py", "smoke-onnx",
    "--model", str(ONNX_MODEL), "--crop-manifest", str(CROP_MANIFEST),
    "--crop-root", str(CROP_ROOT), "--report", str(ONNX_SMOKE_REPORT),
    "--input-size", "380",
], check=True)
onnx_smoke = json.loads(ONNX_SMOKE_REPORT.read_text(encoding="utf-8"))
print({
    "onnx_cpu_status": onnx_smoke["status"],
    "processing_ms": onnx_smoke["processing_ms"],
    "model_sha256": onnx_smoke["model_sha256"],
})

In [ ]:
# 9. ROC·PR·혼동행렬·열화 결과 그래프와 모델 카드
import matplotlib.pyplot as plt
import numpy as np

curves = metrics["test_video_curves"]
test = metrics["test_video"]
conditions = metrics["condition_test"]
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].plot(curves["roc_fpr"], curves["roc_tpr"], label=f"AUC={test['roc_auc']:.4f}")
axes[0, 0].plot([0, 1], [0, 1], "--", color="gray")
axes[0, 0].set(title="Official Test ROC", xlabel="FPR", ylabel="TPR")
axes[0, 0].legend()

axes[0, 1].plot(curves["pr_recall"], curves["pr_precision"], label=f"AP={test['average_precision']:.4f}")
axes[0, 1].set(title="Official Test Precision-Recall", xlabel="Recall", ylabel="Precision")
axes[0, 1].legend()

matrix = np.asarray([
    [test["true_negative"], test["false_positive"]],
    [test["false_negative"], test["true_positive"]],
])
axes[1, 0].imshow(matrix, cmap="Blues")
for row in range(2):
    for column in range(2):
        axes[1, 0].text(column, row, str(matrix[row, column]), ha="center", va="center")
axes[1, 0].set(title="Video Confusion Matrix", xlabel="Predicted", ylabel="Actual")
axes[1, 0].set_xticks([0, 1], ["real", "fake"])
axes[1, 0].set_yticks([0, 1], ["real", "fake"])

names = list(conditions)
auc_values = [conditions[name]["video"]["roc_auc"] for name in names]
fpr_values = [conditions[name]["video"]["fpr"] for name in names]
x = np.arange(len(names))
axes[1, 1].bar(x - 0.18, auc_values, width=0.36, label="ROC-AUC")
axes[1, 1].bar(x + 0.18, fpr_values, width=0.36, label="FPR")
axes[1, 1].set_xticks(x, names, rotation=25, ha="right")
axes[1, 1].set_ylim(0, 1)
axes[1, 1].set_title("촬영 열화 조건")
axes[1, 1].legend()

fig.tight_layout()
fig.savefig(FIGURE, dpi=160, bbox_inches="tight")
plt.show()

MODEL_CARD.write_text(f'''# EfficientNet-B4 Celeb-DF 연구 모델 카드

- 상태: 연구용 기준선, 운영 미승인
- 입력: 정렬 얼굴 RGB 380×380
- 출력: fake logit; sigmoid 적용값이 딥페이크 점수
- 선택 프레임: {metrics['selected_frames_per_video']}
- 점수 통합: {metrics['selected_aggregation']}
- 연구 기준값: {metrics['selected_threshold']}
- 공식 Test Video ROC-AUC: {test['roc_auc']}
- 공식 Test 실제영상 FPR: {test['fpr']}
- 공식 Test 딥페이크 Recall: {test['recall']}
- 공식 Test 커버리지: {metrics['coverage']['official_test_coverage']}
- Gate 통과: {metrics['research_gate']['overall_pass']}

이 모델은 Celeb-DF와 비상업 연구 조건의 InsightFace 얼굴 검출 전처리를 사용했다. 한국인·최신 생성 방식·실제 웹 검색 영상에 대한 운영 성능과 상용 사용권을 증명하지 않는다. 사람 검토 없이 차단·삭제를 자동 실행하지 않는다.
''', encoding="utf-8")
print({"figure": str(FIGURE), "model_card": str(MODEL_CARD)})

In [ ]:
# 10. 비식별 결과와 비공개 API 모델 묶음 저장
import hashlib
import zipfile

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

SANITIZED_BUNDLE = OUTPUT_ROOT / "celebdf_deepfake_sanitized_results.zip"
PRIVATE_MODEL_BUNDLE = OUTPUT_ROOT / "celebdf_deepfake_private_model.zip"
sanitized_files = (
    INVENTORY,
    PREPROCESS_REPORT,
    TRAIN_REPORT,
    METRICS,
    ONNX_EXPORT_REPORT,
    ONNX_SMOKE_REPORT,
    FIGURE,
    MODEL_CARD,
)
with zipfile.ZipFile(SANITIZED_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sanitized_files:
        archive.write(path, arcname=path.name)
with zipfile.ZipFile(PRIVATE_MODEL_BUNDLE, "w", compression=zipfile.ZIP_STORED) as archive:
    archive.write(CHECKPOINT, arcname=CHECKPOINT.name)
    archive.write(ONNX_MODEL, arcname=ONNX_MODEL.name)
    archive.write(MODEL_CARD, arcname=MODEL_CARD.name)

with zipfile.ZipFile(SANITIZED_BUNDLE) as archive:
    names = set(archive.namelist())
    assert PRIVATE_SCORES.name not in names
    assert CROP_MANIFEST.name not in names
    assert not any(name.endswith((".jpg", ".mp4", ".pt", ".onnx")) for name in names)

print({
    "sanitized_results": str(SANITIZED_BUNDLE),
    "sanitized_sha256": file_sha256(SANITIZED_BUNDLE),
    "private_api_model": str(PRIVATE_MODEL_BUNDLE),
    "private_model_sha256": file_sha256(PRIVATE_MODEL_BUNDLE),
    "raw_faces_in_sanitized_bundle": False,
    "warning": "private model bundle must not be published or committed to GitHub",
})

## 결과 판단

- `official_test_auc ≥ 0.90`
- `official_test_fpr ≤ 0.01`
- 내부 Train/Validation 누수 0건

통과하더라도 연구 기준선이다. 비식별 결과 ZIP만 GitHub Issue/PR에 첨부하고, 얼굴 crop·frame score·checkpoint·ONNX는 비공개로 유지한다.